<a href="https://colab.research.google.com/github/hcship/Pytorch/blob/main/Knowledge_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
token=userdata.get('HF_TOKEN')

In [ ]:
import os
from huggingface_hub import HfApi


In [ ]:
from transformers import AutoTokenizer, AutoModel
tokenizer= AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [ ]:
tokenizer.tokenize("hello my name is harshit. I luv AI",return_tensors="pt")

['hello',
 'my',
 'name',
 'is',
 'harsh',
 '##it',
 '.',
 'I',
 'l',
 '##u',
 '##v',
 'AI']

In [ ]:
tokenizer("hello my name is harshit. I luv AI",return_tensors="pt")

{'input_ids': tensor([[  101, 19082,  1139,  1271,  1110,  8213,  2875,   119,   146,   181,
          1358,  1964, 19016,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

own tokenizer

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
corpus=["i love AI","I am learning AI ","soon i will be doing finetuning"]
tokenizer=Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer=trainers.BpeTrainer(vocab_size=100)
tokenizer.train_from_iterator(corpus, trainer)
tokenizer.save("custom-tokenize")

how to load


In [ ]:
from transformers import PreTrainedTokenizerFast

hf_tokenizer= PreTrainedTokenizerFast(
    tokenizer_file="/content/custom-tokenize",
    unk_token="[UNK]",
    pad_token="[PD]",
)

load model

In [ ]:
model=AutoModel.from_pretrained("bert-base-cased")

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Fine tuning and knowledge distillation**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer,DataCollatorWithPadding
import torch.optim as optim
from datasets import load_dataset
from torch.utils.data import DataLoader

In [ ]:
lr=5e-5
epoch=50
temperature=2.0
batch_size=16
alpha_soft=0.5
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")


load dataset

In [ ]:
data= load_dataset("zeroshot/twitter-financial-news-sentiment")


In [ ]:
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})

In [ ]:
train_data= data["train"].shuffle()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize(example):
  return tokenizer(
      example["text"],
      truncation=True)

In [ ]:

tokenize_train=data["train"].map(tokenize, batched=True, remove_columns=["text"])
tokenize_validation= data["validation"].map(tokenize, batched=True, remove_columns=["text"])

Map:   0%|          | 0/9543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

In [ ]:
tokenize_train

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 9543
})

In [ ]:
tokenize_train = tokenize_train.rename_column("label", "labels")
tokenize_validation = tokenize_validation.rename_column("label", "labels")

In [ ]:
collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

In [ ]:
train_dataloader = DataLoader(
    tokenize_train,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collator
)

validation_dataloader = DataLoader(
    tokenize_validation,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collator
)

Loadin teacher model

In [ ]:
teacher= AutoModelForSequenceClassification.from_pretrained(
    "bert-large-uncased",
    num_labels=3
).to(device)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-large-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading student model

In [ ]:
student= AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


freeze teacher

In [ ]:
for p in teacher.parameters():
  p.requires_grad=False
teacher.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1024,

define loss fn

In [ ]:
ce_loss= nn.CrossEntropyLoss()
kl_loss= nn.KLDivLoss(reduction="batchmean")

In [ ]:
optimizer = optim.AdamW(student.parameters(),lr=5e-5)

In [ ]:
from transformers import get_scheduler

lr_scheduler= get_scheduler(
    name="linear",
   optimizer=optimizer,            # Your optimizer
    num_warmup_steps=100,              # Steps to grow the learning rate
    num_training_steps=len(train_dataloader)*epoch
)

In [ ]:
def distil():

    student.train()

    total_loss = 0

    for batch in train_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Teacher prediction
        with torch.no_grad():
            t_logits = teacher(
                input_ids=input_ids,
                attention_mask=attention
            ).logits

            t_soft = torch.softmax(
                t_logits / temperature,
                dim=-1
            )

        # Student prediction
        s_logits = student(
            input_ids=input_ids,
            attention_mask=attention
        ).logits

        s_soft = torch.log_softmax(
            s_logits / temperature,
            dim=-1
        )

        # Distillation loss
        soft_loss = kl_loss(s_soft, t_soft) * (temperature ** 2)

        # Normal classification loss
        hard_loss = ce_loss(s_logits, labels)

        # Final combined loss
        final_loss = alpha_soft * soft_loss + (1 - alpha_soft) * hard_loss

        optimizer.zero_grad()
        final_loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += final_loss.item()

    return total_loss / len(train_dataloader)



In [ ]:
def evaluate():
    student.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in validation_dataloader:
            ids  = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            lbl  = batch["labels"].to(device)
            out  = student(ids, attention_mask=attn).logits
            pred = out.argmax(dim=1)
            correct += (pred == lbl).sum().item()
            total   += lbl.size(0)
    return round(correct / total * 100, 2)

In [ ]:
for ep in range(1, epoch + 1):
    distil()
    acc = evaluate()
    print(f"Epoch {ep}/{epoch} | Validation Accuracy: {acc}%")

Epoch 1/50 | Validation Accuracy: 84.55%
Epoch 2/50 | Validation Accuracy: 86.47%
Epoch 3/50 | Validation Accuracy: 87.9%
Epoch 4/50 | Validation Accuracy: 87.65%
Epoch 5/50 | Validation Accuracy: 87.44%
Epoch 6/50 | Validation Accuracy: 87.44%
Epoch 7/50 | Validation Accuracy: 87.86%
Epoch 8/50 | Validation Accuracy: 89.03%
Epoch 9/50 | Validation Accuracy: 87.65%
Epoch 10/50 | Validation Accuracy: 87.23%
Epoch 11/50 | Validation Accuracy: 87.23%
Epoch 12/50 | Validation Accuracy: 88.23%
Epoch 13/50 | Validation Accuracy: 87.77%
Epoch 14/50 | Validation Accuracy: 87.77%
Epoch 15/50 | Validation Accuracy: 87.23%
Epoch 16/50 | Validation Accuracy: 87.06%
Epoch 17/50 | Validation Accuracy: 88.07%
Epoch 18/50 | Validation Accuracy: 87.98%
Epoch 19/50 | Validation Accuracy: 87.86%
Epoch 20/50 | Validation Accuracy: 87.35%
Epoch 21/50 | Validation Accuracy: 87.94%
